# **Aprendizaje Semi-Supervisado**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np
import time

In [2]:
data = pd.read_csv('diabetes_012_health_indicators_BRFSS2021.csv', delimiter=',')
print('INFORMACION DE TIPO DE DATOS')
data.info()
print('\nDATOS VACIOS')
print(pd.isnull(data).sum())

INFORMACION DE TIPO DE DATOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236378 entries, 0 to 236377
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_012          236378 non-null  float64
 1   HighBP                236378 non-null  int64  
 2   HighChol              236378 non-null  float64
 3   CholCheck             236378 non-null  int64  
 4   BMI                   236378 non-null  float64
 5   Smoker                236378 non-null  float64
 6   Stroke                236378 non-null  float64
 7   HeartDiseaseorAttack  236378 non-null  float64
 8   PhysActivity          236378 non-null  int64  
 9   Fruits                236378 non-null  int64  
 10  Veggies               236378 non-null  int64  
 11  HvyAlcoholConsump     236378 non-null  int64  
 12  AnyHealthcare         236378 non-null  int64  
 13  NoDocbcCost           236378 non-null  float64
 14  GenHlth               2

In [3]:
target_column = 'Diabetes_012'
X = data.drop(columns=[target_column])
y = data[target_column]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42
)

In [5]:
print("Datos cargados y separados exitosamente.")
print(f"Tamaño de X_train (Entrenamiento): {X_train.shape}")
print(f"Tamaño de X_test (Prueba): {X_test.shape}")

Datos cargados y separados exitosamente.
Tamaño de X_train (Entrenamiento): (177283, 21)
Tamaño de X_test (Prueba): (59095, 21)


In [6]:
scaler = StandardScaler()
scaler.fit(X_train.values)

,copy,True
,with_mean,True
,with_std,True


In [7]:
X_train_scaled = scaler.transform(X_train.values)
X_test_scaled = scaler.transform(X_test.values)

Aplicamos K-Means

In [8]:
k = 50

kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
X_digits_dist = kmeans.fit_transform(X_train_scaled)

In [9]:
print(X_digits_dist.shape)
print(X_digits_dist)

(177283, 50)
[[ 4.88073674  2.90756903  3.93671672 ...  7.06020943  3.49602241
   4.9518998 ]
 [ 6.93669232  5.30906793  5.23163527 ...  7.60381189  6.27547335
   6.72079888]
 [ 5.72650913  6.50041     6.67561162 ...  5.02659271  6.29398727
   5.79521491]
 ...
 [ 4.36349589  4.81317258  4.37973848 ...  7.13521686  2.77899511
   5.38298169]
 [ 9.27199015  9.11520154  8.27744262 ... 10.1171485   9.12916807
   9.29206252]
 [ 6.11458603  3.62813233  3.94038959 ...  7.72560598  4.94673486
   5.93836212]]


In [10]:
idxs = np.argmin(X_digits_dist, axis=0)
X_representative_digits = X_train.values[idxs]

In [11]:
print(X_representative_digits.shape)
print(X_representative_digits)

(50, 21)
[[ 1.  1.  1. ... 10.  5.  5.]
 [ 0.  0.  1. ... 10.  5.  7.]
 [ 0.  0.  1. ...  3.  4.  5.]
 ...
 [ 1.  0.  1. ...  5.  5.  2.]
 [ 1.  1.  1. ... 10.  5.  6.]
 [ 1.  1.  1. ...  9.  5.  6.]]


In [12]:
df_prototipos = pd.DataFrame(
    X_representative_digits, 
    columns=X_train.columns
)

print("--- 50 Muestras Prototipo (Muestras más cercanas a los centroides) ---")


print(df_prototipos.head()) 

print("\n--- Estadísticas de las Muestras Prototipo ---")


print(df_prototipos.mean().sort_values(ascending=False))

--- 50 Muestras Prototipo (Muestras más cercanas a los centroides) ---
   HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  HeartDiseaseorAttack  \
0     1.0       1.0        1.0  31.0     0.0     0.0                   0.0   
1     0.0       0.0        1.0  25.0     0.0     0.0                   0.0   
2     0.0       0.0        1.0  27.0     0.0     0.0                   0.0   
3     0.0       0.0        0.0  27.0     0.0     0.0                   0.0   
4     0.0       1.0        1.0  28.0     0.0     0.0                   0.0   

   PhysActivity  Fruits  Veggies  ...  AnyHealthcare  NoDocbcCost  GenHlth  \
0           0.0     0.0      0.0  ...            1.0          0.0      3.0   
1           1.0     1.0      1.0  ...            1.0          0.0      2.0   
2           1.0     1.0      1.0  ...            1.0          0.0      2.0   
3           1.0     1.0      1.0  ...            1.0          0.0      2.0   
4           1.0     1.0      1.0  ...            1.0          0.0     

Anotamos manualmente las etiquetas.

In [13]:
y_representative_digits = y_train.values[idxs]

In [14]:
print(f"Shape de las etiquetas representativas: {y_representative_digits.shape}")
print(f"Primeras 5 etiquetas de las muestras más importantes: {y_representative_digits[:5]}")

Shape de las etiquetas representativas: (50,)
Primeras 5 etiquetas de las muestras más importantes: [0. 0. 0. 0. 0.]


Entrenamos un clasificador.

In [15]:
X_representative_digits_scaled = scaler.transform(X_representative_digits)

In [16]:
log_reg2 = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)

log_reg2.fit(X_representative_digits_scaled, y_representative_digits)
accuracy = log_reg2.score(X_test_scaled, y_test)

print(f"\nPrecisión del modelo entrenado con 50 muestras (vs {len(y_test)} en el test set): {accuracy:.4f}")


Precisión del modelo entrenado con 50 muestras (vs 59095 en el test set): 0.7791


In [17]:
log_reg = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
%time log_reg.fit(X_train_scaled[:50], y_train[:50])
log_reg.score(X_test_scaled, y_test)

CPU times: total: 15.6 ms
Wall time: 5.89 ms


0.7735679837549708

Anotamos de manera automatica.

In [18]:
y_train_propagated = np.empty(len(X_train))
for i in range(k):
  y_train_propagated[kmeans.labels_==i] = y_representative_digits[i]

In [19]:
log_reg3 = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
log_reg3.fit(X_train_scaled[:1000], y_train_propagated[:1000])
log_reg3.score(X_test_scaled, y_test)

0.8086132498519333

# **Aprendizaje Activo**

In [20]:
probas = log_reg3.predict_proba(X_train_scaled[:1000])
labels_ixs = np.argmax(probas, axis=1)
labels = np.array([proba[ix] for proba, ix in zip(probas, labels_ixs)])
sorted_ixs = np.argsort(labels)
labels[sorted_ixs[:10]]

array([0.50022804, 0.50261427, 0.50450925, 0.50725153, 0.51639902,
       0.52433147, 0.52487639, 0.52705578, 0.53077893, 0.53199955])

In [21]:
X_lowest_confidence_samples = X_train.values[:1000][sorted_ixs[:k]]

df_incertidumbre = pd.DataFrame(
    X_lowest_confidence_samples, 
    columns=X_train.columns
)

print("--- 50 Muestras con MÁXIMA Incertidumbre del Modelo ---")
print(df_incertidumbre.head()) 

--- 50 Muestras con MÁXIMA Incertidumbre del Modelo ---
   HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  HeartDiseaseorAttack  \
0     1.0       1.0        1.0  30.0     1.0     0.0                   1.0   
1     0.0       0.0        1.0  22.0     1.0     0.0                   0.0   
2     0.0       0.0        1.0  48.0     1.0     0.0                   0.0   
3     1.0       1.0        1.0  40.0     1.0     0.0                   1.0   
4     1.0       1.0        1.0  27.0     0.0     0.0                   0.0   

   PhysActivity  Fruits  Veggies  ...  AnyHealthcare  NoDocbcCost  GenHlth  \
0           0.0     1.0      0.0  ...            1.0          0.0      4.0   
1           1.0     0.0      0.0  ...            1.0          0.0      3.0   
2           0.0     0.0      1.0  ...            1.0          1.0      3.0   
3           1.0     0.0      1.0  ...            1.0          1.0      3.0   
4           0.0     1.0      1.0  ...            1.0          1.0      4.0   

   Men

In [22]:
y_lowest = y_train.values[:1000][sorted_ixs[:k]]
y_lowest

array([0., 0., 0., 0., 2., 2., 0., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 2., 1., 0., 0., 0., 2., 2., 2., 0., 2., 2., 0.,
       0., 2., 0., 0., 2., 0., 0., 1., 2., 2., 0., 0., 0., 0., 0., 2.])

In [23]:
y_train2 = y_train_propagated[:1000].copy()
y_train2[sorted_ixs[:k]] = y_lowest

In [24]:
log_reg5 = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
%time log_reg5.fit(X_train_scaled[:1000], y_train2)
log_reg5.score(X_test_scaled, y_test)

CPU times: total: 78.1 ms
Wall time: 22.5 ms


0.8106438785007192